# End of week 1 exercise

## Dynamically pick an LLM provider to let MathXpert answer your math questions.

In [ ]:
import os
import logging
from enum import StrEnum
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, clear_output, Latex

load_dotenv(override=True)

In [ ]:
logging.basicConfig(level=logging.DEBUG)
logger = logging


for package in ['openai', 'httpcore', 'httpx']:
    logging.getLogger(package).setLevel(logging.ERROR)

## Free Cloud Providers

Grab your free API Keys from these generous sites:

- https://openrouter.ai/
- https://ollama.com/

In [ ]:
class Provider(StrEnum):
    OLLAMA = 'Ollama'
    OPENROUTER = 'OpenRouter'

models: dict[Provider, list[str]] = {
    Provider.OLLAMA: [],
    Provider.OPENROUTER: [],
}
providers: dict[Provider, OpenAI] = {}

def get_api_key_in_collab(env_name: str) -> str:
    try:
      from google.colab import userdata
      return userdata.get(env_name)
    except Exception:
      return ''
      

def get_api_key(env_name: str) -> str:
    '''Gets the value from the environment, otherwise ask the user for it if not set'''
    key = os.environ.get(env_name) or get_api_key_in_collab(env_name)

    if not key:
        key = getpass(f'Enter {env_name}:').strip()

    if key:
        logger.info(f'✅ {env_name} provided')
    else:
        logger.warning(f'❌ {env_name} not provided')
    return key


if api_key := get_api_key('OLLAMA_API_KEY'):
    providers[Provider.OLLAMA] = OpenAI(base_url='https://ollama.com/v1', api_key=api_key)

if api_key := get_api_key('OPENROUTER_API_KEY'):
    providers[Provider.OPENROUTER] = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)

In [ ]:
def get_messages(question: str) -> list[dict[str, str]]:
    """Generate messages for the chat models."""

    system_prompt = '''
    You are MathXpert, an expert Mathematician who makes math fun to learn by relating concepts to real 
    practical usage to whip up the interest in learners.
    
    Explain step-by-step thoroughly how to solve a math problem. Respond in **LaTex**'
    '''

    return [
        {'role': 'system', 'content': system_prompt },
        {'role': 'user', 'content': question},
    ]

In [ ]:
selected_provider, selected_model, client = '', '', None

try:
    selected_provider, client = next(iter(providers.items()))
except Exception:
    logger.warning(f'❌ no provider configured and everything else from here will FAIL 🤦, I know you know this already.')

def load_models_if_needed(client: OpenAI, selected_provider):
    global selected_model, models

    if client and not models.get(selected_provider):
        logging.info(f'📡 Fetching {selected_provider} models...')
        
        models[selected_provider] = [model.id for model in client.models.list()]
        provider_models = models[selected_provider]
        selected_model = provider_models[0] if provider_models else ''

def on_provider_change(change):
    global selected_provider, client, models

    selected_provider = change['new']
    client = providers.get(selected_provider)
    load_models_if_needed(client, selected_provider)

    model_selector.options = models.get(selected_provider, [])

def on_model_change(change):
    global selected_model

    selected_model = change['new']
    logger.info(f'👉 Selected model: {selected_model}')

provider_selector = widgets.Dropdown(
    options=list(providers.keys()),
    description='Select an LLM provider:',
    style={'description_width': 'initial'},
)


load_models_if_needed(client, selected_provider)

model_selector = widgets.Dropdown(
    options=models.get(selected_provider, []),
    description='Model:',
    style={'description_width': 'initial'},
)

provider_selector.observe(on_provider_change, names='value')
model_selector.observe(on_model_change, names='value')

logger.info(f'ℹ️ Provider: {selected_provider} Model: {selected_model}, Client: {client}')

In [ ]:
handle = display(None, display_id=True)

def ask(client: OpenAI | None, model: str, question: str):
    if client is None:
        logger.warning('You should have provided the API Keys you know. Fix 🔧 this and try again ♻️.')
        return

    try:
        prompt = get_messages(question=question)
        response = client.chat.completions.create(
            model=model,
            messages=prompt,
            stream=True,
        )
    
        output = ''
        for chunk in response:
            output  += chunk.choices[0].delta.content or ''
            handle.update(Latex(output))
    except Exception as e:
        clear_output(wait=True) 
        logger.error(f'🔥 An error occurred: {e}')

In [ ]:
display(widgets.HBox([provider_selector, model_selector]))

In [ ]:
input_label = "Ask your question (Type 'q' to quit): "
question = input(input_label)

while question.strip().lower() not in ['quit', 'q']:
    clear_output(wait=True)
    logger.info(f'ℹ️ Provider: {selected_provider} Model: {selected_model}, Client: {client}')
    print(f'Question: {question}')
    # model = selected_model
    ask(client, selected_model, question)

    question = input(input_label)